# 06 - Building the RAG Chain with LangChain

This is the **keystone notebook** — we wire together everything from the previous notebooks into a complete RAG pipeline using **LCEL (LangChain Expression Language)**.

## LCEL Primer

LCEL uses the pipe operator (`|`) to chain components:

```python
chain = retriever | format_docs | prompt | llm | output_parser
```

Each component is a **Runnable** — it takes input, does something, and passes output to the next component. This is LangChain's declarative way to build pipelines.

In [ ]:
import os
import sys
import tempfile

sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))

from dotenv import load_dotenv

load_dotenv(os.path.join(os.path.dirname(os.getcwd()), ".env"))

## Step 1: Build the Knowledge Base

Load documents, chunk them, and store in ChromaDB.

In [ ]:
from rag_engine.chunking.strategies import chunk_documents
from rag_engine.loaders import load_documents
from rag_engine.vectorstore.chroma_store import add_documents

DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "data", "sample")

# Load all sample documents
md_docs = load_documents(os.path.join(DATA_DIR, "rag_survey.md"))
csv_docs = load_documents(os.path.join(DATA_DIR, "ml_glossary.csv"))
all_docs = md_docs + csv_docs

# Chunk
chunks = chunk_documents(all_docs, strategy="recursive", chunk_size=512, chunk_overlap=50)
print(f"Total chunks: {len(chunks)}")

# Store in ChromaDB
temp_dir = tempfile.mkdtemp()
vectorstore = add_documents(chunks, collection_name="nb06", persist_directory=temp_dir)
print("Knowledge base ready.")

## Step 2: Create the Retriever

In [ ]:
from rag_engine.retrieval.retriever import RetrieverFactory

retriever = RetrieverFactory.create_retriever(vectorstore, strategy="mmr", top_k=5)

## Step 3: Create the LLM

In [ ]:
from rag_engine.llm.provider import LLMProvider

llm = LLMProvider.get_llm(temperature=0.0)
print(f"Using LLM: {llm.__class__.__name__}")

## Step 4: Build the RAG Chain

Here's where LCEL shines. Our `build_rag_chain` function composes:

```
question → retriever → format_documents → prompt → llm → string output
```

In [ ]:
from rag_engine.chains.rag_chain import build_rag_chain

rag_chain = build_rag_chain(retriever, llm)

# Let's see what the chain looks like
print("Chain structure:")
print(rag_chain)

## Step 5: Ask Questions!

Now we can query our knowledge base conversationally.

In [ ]:
questions = [
    "What is RAG and what problem does it solve?",
    "What are the different chunking strategies?",
    "How does MMR retrieval differ from similarity search?",
    "What is HyDE and how does it improve retrieval?",
]

for q in questions:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    print(f"{'='*60}")
    answer = rag_chain.invoke(q)
    print(f"A: {answer}")

## Step 6: RAG Chain with Source Documents

In production, you often want to show users **where** the answer came from. Our `build_rag_chain_with_sources` variant returns both the answer and the source documents.

In [ ]:
from rag_engine.chains.rag_chain import build_rag_chain_with_sources

chain_with_sources = build_rag_chain_with_sources(retriever, llm)

result = chain_with_sources.invoke({"question": "What are the RAGAS evaluation metrics?"})

print("Answer:")
print(result["answer"])
print(f"\n\nSources ({len(result['source_documents'])} documents):")
for i, doc in enumerate(result["source_documents"], 1):
    print(f"  [{i}] {doc.metadata.get('source', 'N/A')} — {doc.page_content[:80]}...")

## Understanding the Prompt

The prompt template is critical for RAG quality. Let's look at what we're sending to the LLM.

In [ ]:
from rag_engine.chains.prompts import RAG_PROMPT

print("Prompt template messages:")
for msg in RAG_PROMPT.messages:
    print(f"\n  Role: {msg.__class__.__name__}")
    if hasattr(msg, 'prompt'):
        print(f"  Template: {msg.prompt.template[:200]}...")

Key prompt engineering decisions for RAG:

1. **"Use ONLY the provided context"** — prevents the LLM from using its own knowledge (reduces hallucination)
2. **"If the context doesn't contain enough information, say so"** — honest uncertainty over fabrication
3. **"Reference which source document"** — enables citation, builds trust

## The Full Pipeline in ~15 Lines

Here's the complete RAG pipeline condensed:

In [ ]:
# The complete RAG pipeline
from rag_engine.chains.rag_chain import build_rag_chain
from rag_engine.chunking.strategies import chunk_documents
from rag_engine.llm.provider import LLMProvider
from rag_engine.loaders import load_documents
from rag_engine.retrieval.retriever import RetrieverFactory
from rag_engine.vectorstore.chroma_store import add_documents

# 1. Ingest
docs = load_documents("path/to/your/documents.md")
chunks = chunk_documents(docs, chunk_size=512)
vs = add_documents(chunks)

# 2. Query
retriever = RetrieverFactory.create_retriever(vs, strategy="mmr")
llm = LLMProvider.get_llm()
chain = build_rag_chain(retriever, llm)
answer = chain.invoke("Your question here")

## Summary

We built a complete RAG pipeline:

1. **Load** documents from any source
2. **Chunk** them into retrieval-sized pieces
3. **Embed** and store in ChromaDB
4. **Retrieve** relevant chunks for a query
5. **Generate** a grounded answer with an LLM

The key LangChain concept is **LCEL composition**: each component is a Runnable, and the pipe operator chains them together into a clean, declarative pipeline.

**Next:** [07 - Swappable LLM Providers](./07_swappable_llm_providers.ipynb) — running the same pipeline with OpenAI and Anthropic.